In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Load dataset (no header in file)
cols = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "target",
]

df = pd.read_csv("DSBDAL_datasets/heart.csv", header=None, names=cols)

# a) Data cleaning: remove NA, ?, negative values

df = df.replace("?", np.nan)
for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df.loc[df[col] < 0, col] = np.nan

df = df.dropna()

# b) Error correcting: simple outlier removal (IQR) on numeric features

def iqr_filter(data, col):
    q1 = data[col].quantile(0.25)
    q3 = data[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    return data[(data[col] >= low) & (data[col] <= high)]

num_cols = [c for c in cols if c != "target"]
for c in num_cols:
    df = iqr_filter(df, c)

# c) Data transformation: scale features

X = df.drop(columns=["target"])
target_bin = (df["target"] > 0).astype(int)

# d) Models: Logistic Regression and kNN

X_train, X_test, y_train, y_test = train_test_split(
    X, target_bin, test_size=0.2, random_state=42, stratify=target_bin
)

log_reg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))

log_reg.fit(X_train, y_train)
knn.fit(X_train, y_train)

pred_lr = log_reg.predict(X_test)
pred_knn = knn.predict(X_test)

print("Logistic Regression accuracy:", accuracy_score(y_test, pred_lr))
print("kNN accuracy:", accuracy_score(y_test, pred_knn))